# Train Multichannel Sarcasm Detection — Google Colab

**Cara pakai:**
1. Pastikan folder `multi_channel_method/` sudah di-upload ke Google Drive (`My Drive/multi_channel_method/`)
2. Pilih **Runtime → Change runtime type → T4 GPU**
3. Jalankan cell dari atas ke bawah satu per satu

## Cell 1 — Environment Setup (Colab / Kaggle)

> **Kaggle**: ubah  sesuai nama dataset di   
> **Colab**: tidak perlu mengubah apapun, Drive akan di-mount otomatis


In [ ]:
import os, sys, shutil

# Auto-detect environment: Kaggle vs Colab
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    # ── KAGGLE SETUP ──────────────────────────────────────────────────────
    # Cari nama folder dataset di /kaggle/input/ lalu isi KAGGLE_INPUT_PATH
    print("Input datasets tersedia:", os.listdir('/kaggle/input'))

    # Ubah KAGGLE_INPUT_PATH sesuai nama dataset kamu
    KAGGLE_INPUT_PATH = '/kaggle/input/NAMA-DATASET/multi_channel_method'
    WORK_DIR = '/kaggle/working/multi_channel_method'

    if not os.path.exists(WORK_DIR):
        print(f'Menyalin project ke {WORK_DIR} ...')
        shutil.copytree(KAGGLE_INPUT_PATH, WORK_DIR, dirs_exist_ok=False)
        print('Selesai.')
    else:
        print(f'{WORK_DIR} sudah ada, skip copy.')

    os.chdir(WORK_DIR)
    print(f'Kaggle mode: working dir = {WORK_DIR}')

else:
    # ── COLAB SETUP ───────────────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/multi_channel_method'
    os.chdir(PROJECT_PATH)
    print(f'Colab mode: working dir = {PROJECT_PATH}')

print('Files:', os.listdir('.'))


## Cell 2 — Konfirmasi Working Directory


In [ ]:
import os
print('Working dir:', os.getcwd())
print('Files found:', os.listdir('.'))


## Cell 3 — Install / Upgrade Dependencies & Cek GPU

In [ ]:
!pip install -q --upgrade transformers sentencepiece
!nvidia-smi

## Cell 4 — Verifikasi File Data

In [ ]:
import os

all_ok = True
for dataset in ['reddit', 'twitter']:
    for split in ['train', 'test', 'validation']:
        for ext in ['.csv', '.csv.graph.new', '.csv.sentic']:
            path = f'real_data/{dataset}/{split}{ext}'
            status = 'OK' if os.path.exists(path) else 'MISSING!'
            if status == 'MISSING!':
                all_ok = False
            print(f'[{dataset}] {split}{ext}: {status}')

print()
if all_ok:
    print('Semua file tersedia. Siap training!')
else:
    print('Ada file yang hilang! Periksa upload Google Drive.')

## Cell 5 — Training: Dataset Reddit

Output log juga disimpan ke `logs/colab_reddit.log`.

In [ ]:
import os
os.makedirs('logs', exist_ok=True)
os.makedirs('checkpoints/reddit', exist_ok=True)

In [ ]:
!python train_multichannel.py \
    --train_data       real_data/reddit/train.csv \
    --test_data        real_data/reddit/test.csv \
    --val_data         real_data/reddit/validation.csv \
    --graph_dir        real_data/reddit/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --epochs           100 \
    --batch_size       32 \
    --dataset_name     reddit \
    --data_dir         real_data/reddit/ \
    --checkpoint_dir   checkpoints/reddit/ \
    --seed             42 \
    2>&1 | tee logs/colab_reddit.log

## Cell 6 — Training: Dataset Twitter

Output log juga disimpan ke `logs/colab_twitter.log`.

In [ ]:
os.makedirs('checkpoints/twitter', exist_ok=True)

In [ ]:
!python train_multichannel.py \
    --train_data       real_data/twitter/train.csv \
    --test_data        real_data/twitter/test.csv \
    --val_data         real_data/twitter/validation.csv \
    --graph_dir        real_data/twitter/ \
    --train_split_name train.csv \
    --test_split_name  test.csv \
    --val_split_name   validation.csv \
    --epochs           100 \
    --batch_size       32 \
    --dataset_name     twitter \
    --data_dir         real_data/twitter/ \
    --checkpoint_dir   checkpoints/twitter/ \
    --seed             42 \
    2>&1 | tee logs/colab_twitter.log

## Cell 7 (Opsional) — Lihat Log Hasil Training

In [ ]:
# Tampilkan 50 baris terakhir log Reddit
!tail -50 logs/colab_reddit.log

In [ ]:
# Tampilkan 50 baris terakhir log Twitter
!tail -50 logs/colab_twitter.log